# Enterprise Architecure

```text
                    ┌─────────────────────┐
                    │ Company PDF Files   │
                    │                     │
                    │ employee_handbook   │
                    │ IT_policy           │
                    │ security_policy     │
                    │ leave_policy        │
                    │ onboarding          │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Document Processing │
                    │                     │
                    │ PDF Loader          │
                    │ Text Extraction     │
                    │ Cleaning            │
                    │ Metadata            │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Chunking            │
                    │                     │
                    │ 300-500 tokens      │
                    │ overlap 50 tokens   │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Embedding Model     │
                    │                     │
                    │ text → vector       │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Pinecone            │
                    │                     │
                    │ Vector              │
                    │ Metadata            │
                    └──────────┬──────────┘
                               │
                       User Question
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Query Embedding     │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Metadata Filtering  │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Vector Search       │
                    │ top_k = 20          │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Reranker            │
                    │ top 20 → top 5      │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Prompt Construction │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ LLM                 │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ Answer + Citations  │
                    └─────────────────────┘
```

# Project Structure

```text
company-rag/
│
├── documents/
│   ├── employee_handbook.pdf
│   ├── IT_policy.pdf
│   ├── security_policy.pdf
│   ├── leave_policy.pdf
│   └── onboarding.pdf
│
├── .env
├── config.py
├── ingest.py
├── retrieve.py
├── rag.py
└── evaluate.py
```

# Document Processing

## Load pdf

In [1]:
# ingest.py

from pathlib import Path

from pypdf import PdfReader


def load_pdf(pdf_path: str) -> list[dict]:
    reader = PdfReader(pdf_path)

    documents = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()

        if not text:
            continue

        documents.append(
            {
                "text": text,
                "page": page_number,
                "source": Path(pdf_path).name,
            }
        )

    return documents

In [2]:
pages = load_pdf("documents/leave_policy.pdf")
print(pages[0])

{'text': "Southern Cross Technology Pty Ltd\nPage 1\n Leave Policy\nGuidance for annual leave, personal/carer's leave, compassionate leave and other approved absences.\nInternal Company Document | Version 1.0 | Effective: 1 September 2026\n1. General Principles\n\x7f Employees should request planned leave as early as practical so teams can manage workload and coverage.\n\x7f Leave approval is subject to applicable employment conditions, operational requirements and relevant Australian\nworkplace law.\n\x7f The HR system is the source of record for approved leave balances and requests.\n2. Annual Leave\n\x7f Full-time and part-time employees accrue annual leave according to their applicable employment conditions.\n\x7f Employees should submit planned annual leave through the HR system before making non-refundable travel\narrangements.\n\x7f Managers consider team coverage, business needs and the employee's accrued balance when reviewing\nrequests.\n3. Personal and Carer's Leave\n\x7f Pe

## Chunking

In [3]:
# ingest.py
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [4]:
text = """
Full-time employees receive 20 days of annual leave
per year. Employees should submit leave requests
through the HR system.
"""

chunks = chunk_text(text)

for chunk in chunks:
    print(chunk)

Full-time employees receive 20 days of annual leave per year. Employees should submit leave requests through the HR system.


## Add Metadata

In [5]:
def create_chunks(pdf_path: str) -> list[dict]:

    pages = load_pdf(pdf_path)

    results = []

    for page in pages:
        chunks = chunk_text(
            page["text"],
            chunk_size=500,
            overlap=50,
        )

        for index, chunk in enumerate(chunks):
            results.append(
                {
                    "text": chunk,
                    "metadata": {
                        "source": page["source"],
                        "page": page["page"],
                        "chunk_id": index,
                        "document_type": "company_policy",
                        "department": "HR",
                        "access_level": "employee",
                        "version": "2026.1",
                    },
                }
            )

    return results

In [6]:
chunks = create_chunks("documents/leave_policy.pdf")
print(chunks[0])

{'text': "Southern Cross Technology Pty Ltd Page 1 Leave Policy Guidance for annual leave, personal/carer's leave, compassionate leave and other approved absences. Internal Company Document | Version 1.0 | Effective: 1 September 2026 1. General Principles \x7f Employees should request planned leave as early as practical so teams can manage workload and coverage. \x7f Leave approval is subject to applicable employment conditions, operational requirements and relevant Australian workplace law. \x7f The HR system is the source of record for approved leave balances and requests. 2. Annual Leave \x7f Full-time and part-time employees accrue annual leave according to their applicable employment conditions. \x7f Employees should submit planned annual leave through the HR system before making non-refundable travel arrangements. \x7f Managers consider team coverage, business needs and the employee's accrued balance when reviewing requests. 3. Personal and Carer's Leave \x7f Personal/carer's lea

# Embedding

In [7]:
from config import EMBEDDING_MODEL
from openai import OpenAI

client = OpenAI()


def create_embedding(text: str) -> list[float]:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )

    return response.data[0].embedding

In [8]:
vector = create_embedding("How many days of annual leave do employees receive?")

print(len(vector))

1536


# Pinecone

In [9]:
from config import PINECONE_INDEX_NAME
from pinecone import Pinecone, ServerlessSpec

print(PINECONE_INDEX_NAME)

need_to_delete_indexes = [
    PINECONE_INDEX_NAME,
]

pc = Pinecone()

pinecone_indexes = pc.list_indexes()

for to_delete_index in need_to_delete_indexes:
    if to_delete_index in [index.name for index in pinecone_indexes]:
        pc.delete_index(to_delete_index)
        print(f"{to_delete_index} successfully deleted.")
    else:
        print(f"{to_delete_index} not in index list.")

# Create your Pinecone index
pc.create_index(
    name=PINECONE_INDEX_NAME,
    dimension=1536,
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    ),
)


index = pc.Index(PINECONE_INDEX_NAME)

# Print the index statistics
print(index.describe_index_stats())

company-knowledge-rag
company-knowledge-rag not in index list.
DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)
